# Experiment 17 — Real stratification (day-flags only) vs default: BayesHalvingSearchCV vs PatternSearchCV (2026-07-19)

Every prior experiment's `subsample='stratified'` watched *all* columns, including the near-continuous weather columns - which makes every row unique and silently degrades the sampler to pure bit-reversal (1.0 rows/run, the transition-detection/alternating logic never fires). This run fixes that: `subsample_columns` is set to the three genuinely low-cardinality day-varying flags (`HasPromotions`, `IsHoliday`, `IsOpen`), which measured 2.5 rows/run avg - the alternating/novel-seat logic actually engages now.

Default ladder (`data_zones=(0.005, 0.01, 0.1, 1.0)`), official grid, `TimeSeriesSplit(5)`, MAE, `n_starts=1`, `random_state=0`, `verbose=2` on both arms (adopting verbose=2 as standard going forward). Data loaded from the saved processed CSV (`train_processed_dayflags_stratify.csv`) rather than re-running the raw pipeline.

Directly comparable to Experiment 13 (same default ladder, same grid, same estimators) - the only difference is a real vs degenerate stratified sampler.

In [1]:
# --- load the saved, already-processed training set (skip re-deriving) ---
import time
import numpy as np
import pandas as pd

t0 = time.time()
trainBench = pd.read_csv(r"C:\FILES\Code\Benchmarking\train_processed_dayflags_stratify.csv")
mask = trainBench.columns.difference(["NumberOfSales"])
trainDataset_X = trainBench[mask]
trainDataset_y = trainBench["NumberOfSales"]
print(f"loaded {trainBench.shape} in {time.time() - t0:.1f}s")
print(list(mask))

loaded (418416, 31) in 7.0s
['AssortmentType', 'Date', 'Events', 'HasPromotions', 'IsHoliday', 'IsOpen', 'Max_Dew_PointC', 'Max_Humidity', 'Max_Sea_Level_PressurehPa', 'Max_TemperatureC', 'Max_Wind_SpeedKm_h', 'Mean_Dew_PointC', 'Mean_Humidity', 'Mean_Sea_Level_PressurehPa', 'Mean_TemperatureC', 'Mean_Wind_SpeedKm_h', 'Min_Dew_PointC', 'Min_Humidity', 'Min_Sea_Level_PressurehPa', 'Min_TemperatureC', 'NearestCompetitor', 'NumberOfCustomers', 'Precipitationmm', 'Region', 'Region_AreaKM2', 'Region_GDP', 'Region_PopulationK', 'StoreID', 'StoreType', 'WindDirDegrees']


In [2]:
# --- common setup: default ladder, real stratification via subsample_columns ---
import numpy as np
from collections import Counter
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.model_selection import TimeSeriesSplit
from pattern_search_cv import BayesHalvingSearchCV, PatternSearchCV

X, y = trainDataset_X, trainDataset_y
N_features = X.shape[1]
ZONES = [0.005, 0.01, 0.1, 1.0]  # package default

DAY_FLAGS = ['HasPromotions', 'IsHoliday', 'IsOpen']
SUBSAMPLE_COLUMNS = [list(mask).index(c) for c in DAY_FLAGS]
print(f"subsample_columns={SUBSAMPLE_COLUMNS} ({DAY_FLAGS})")

def make_clf():
    return ExtraTreesRegressor(n_estimators=240, max_features=max(1, N_features - 15),
                               max_depth=16, n_jobs=1, random_state=0)

param_grid = {
    "max_features": [2, 3, 4],
    "n_estimators": list(range(10, 261, 10)),
    "max_depth": [5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17],
}
tscv = TimeSeriesSplit(n_splits=5)


def _summarize(arm, search, wall):
    res = search.cv_results_
    n_res = np.asarray(res["n_resources"])
    tiers = Counter(int(v) for v in n_res)
    fit_work_total = float(np.sum(np.asarray(res["mean_fit_time"]) * 5))
    out = {
        "arm": arm, "wall": wall,
        "n_fits": len(res["params"]),
        "tiers": tiers,
        "equiv": float(np.sum(n_res / len(y))),
        "best": search.best_params_, "best_mae": -search.best_score_,
        "fit_work_total": fit_work_total,
    }
    print(f"\n{arm:22s}: {out['n_fits']} evals, {out['equiv']:.3f} equiv, "
         f"{wall:.1f} s wall, best {out['best']} MAE {out['best_mae']:.3f}")
    for n_rows, count in sorted(tiers.items()):
        print(f"    {count:3d} fits @ {n_rows:>7d} rows ({n_rows / len(y):.4%})")
    return out

subsample_columns=[3, 4, 5] (['HasPromotions', 'IsHoliday', 'IsOpen'])


In [3]:
search_psc = PatternSearchCV(make_clf(), param_grid,
                             scoring="neg_mean_absolute_error", cv=tscv,
                             n_jobs=-1, data_zones=ZONES,
                             subsample="stratified", subsample_columns=SUBSAMPLE_COLUMNS,
                             random_state=0, verbose=2)
t0 = time.time()
search_psc.fit(X, y)
wall_psc = time.time() - t0
PSC = _summarize("PatternSearchCV (patient, defaults)", search_psc, wall_psc)

[pattern_search_cv] PatternSearchCV: optimizing metric = neg_mean_absolute_error


[pattern_search_cv] cv = TimeSeriesSplit


[pattern_search_cv]   max_features : [2, 3, 4]


[pattern_search_cv]   n_estimators : [10, 20, 30, 40, 50, 60, 70, 80, 90, 100, 110, 120, 130, 140, 150, 160, 170, 180, 190, 200, 210, 220, 230, 240, 250, 260]


[pattern_search_cv]   max_depth : [5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17]


...

[pattern_search_cv] fit_time per fold: [ 23.56267214  47.17504144  70.06180573  97.83008194 124.79306126]


[pattern_search_cv] fit_time: 72.684532


[pattern_search_cv] score_time per fold: [3.60232091 3.53915739 3.66364455 3.38921952 3.47580147]


[pattern_search_cv] score_time: 3.534029



PatternSearchCV (patient, defaults): 22 evals, 5.085 equiv, 1481.6 s wall, best {'max_features': 4, 'n_estimators': 130, 'max_depth': 17} MAE 805.038
     17 fits @    2093 rows (0.5002%)
      5 fits @  418416 rows (100.0000%)


In [4]:
search_bhs = BayesHalvingSearchCV(make_clf(), param_grid,
                                  scoring="neg_mean_absolute_error", cv=tscv,
                                  n_jobs=-1, data_zones=ZONES,
                                  subsample="stratified", subsample_columns=SUBSAMPLE_COLUMNS,
                                  random_state=0, verbose=2, n_iter=25, promote_k=3)
t0 = time.time()
search_bhs.fit(X, y)
wall_bhs = time.time() - t0
BHS = _summarize("BayesHalvingSearchCV", search_bhs, wall_bhs)
BHS["n_cache_hits"] = search_bhs.n_cache_hits_

[pattern_search_cv] BayesHalvingSearchCV: optimizing metric = neg_mean_absolute_error


[pattern_search_cv] cv = TimeSeriesSplit


[pattern_search_cv]   max_features : [2, 3, 4]


[pattern_search_cv]   n_estimators : [10, 20, 30, 40, 50, 60, 70, 80, 90, 100, 110, 120, 130, 140, 150, 160, 170, 180, 190, 200, 210, 220, 230, 240, 250, 260]


[pattern_search_cv]   max_depth : [5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17]


...

[pattern_search_cv] fit_time per fold: [ 23.48509097  46.41871786  72.51980925  96.55308294 122.44305634]


[pattern_search_cv] fit_time: 72.283951


[pattern_search_cv] score_time per fold: [3.55952573 3.65128589 3.75486588 3.39774871 3.44881439]


[pattern_search_cv] score_time: 3.562448



BayesHalvingSearchCV  : 28 evals, 3.125 equiv, 1280.6 s wall, best {'max_features': 4, 'n_estimators': 130, 'max_depth': 17} MAE 805.038
     25 fits @    2093 rows (0.5002%)
      3 fits @  418416 rows (100.0000%)


In [5]:
# --- comparison table, fits-per-tier rows included ---
arms = [PSC, BHS]
all_tiers = sorted(set().union(*(a['tiers'].keys() for a in arms)))

print("=" * 100)
print(f"zones ladder {ZONES} (real stratification: subsample_columns={DAY_FLAGS})")
cols = [a['arm'] for a in arms]
print(f"{'':24s}" + "".join(f"{c:>26s}" for c in cols))
print(f"{'total evaluations':24s}" + "".join(f"{a['n_fits']:>26d}" for a in arms))
for n_rows in all_tiers:
    frac = n_rows / len(y)
    row_label = f'fits @ {frac:.4%}'
    print(f"{row_label:24s}" + "".join(f"{a['tiers'].get(n_rows, 0):>26d}" for a in arms))
print(f"{'full-fit equivalents':24s}" + "".join(f"{a['equiv']:>26.3f}" for a in arms))
print(f"{'wall-clock (s)':24s}" + "".join(f"{a['wall']:>26.1f}" for a in arms))
print(f"{'summed fit work (s)':24s}" + "".join(f"{a['fit_work_total']:>26.1f}" for a in arms))
print(f"{'best point':24s}" + "".join(
    f"{str((a['best']['max_features'], a['best']['n_estimators'], a['best']['max_depth'])):>26s}"
    for a in arms))
print(f"{'best CV MAE':24s}" + "".join(f"{a['best_mae']:>26.3f}" for a in arms))
print()
print(f"BHS cache hits: {BHS['n_cache_hits']}")
print()
print(f"reference: Experiment 13 (degenerate/all-column stratification, same ladder):")
print(f"  PatternSearchCV: 22 evals, 5.09 equiv, best (4,130,17) MAE 805.038")
print(f"  BayesHalvingSearchCV: 28 evals, 3.89 equiv, best (4,150,17) MAE 805.730")

zones ladder [0.005, 0.01, 0.1, 1.0] (real stratification: subsample_columns=['HasPromotions', 'IsHoliday', 'IsOpen'])
                        PatternSearchCV (patient, defaults)      BayesHalvingSearchCV
total evaluations                               22                        28
fits @ 0.5002%                                  17                        25
fits @ 100.0000%                                 5                         3
full-fit equivalents                         5.085                     3.125
wall-clock (s)                              1481.6                    1280.6
summed fit work (s)                         2712.3                    2080.5
best point                            (4, 130, 17)              (4, 130, 17)
best CV MAE                                805.038                   805.038

BHS cache hits: 0

reference: Experiment 13 (degenerate/all-column stratification, same ladder):
  PatternSearchCV: 22 evals, 5.09 equiv, best (4,130,17) MAE 805.038
  BayesHalvi

## Summary

Fill in after running: does real stratification (vs the degenerate all-column version) change the optimum found, the MAE, the number of evaluations, or the full-fit equivalents for either estimator, compared to Experiment 13?